## 📊 Part 1: Dataset Selection & Initial Inspection
In this section, we choose our Student Performance dataset, identify our target variable (`G3`), and inspect the data's raw structure using basic shape and summary statistics.

In [ ]:
import pandas as pd

df = pd.read_csv("student-mat.csv", sep=";")

print("Dataset Shape (Rows, Columns):", df.shape)
print("-" * 50)

print("--- FIRST 5 ROWS  ---")
display(df.head())
print("-" * 50)

print("--- DATASET SUMMARY ---")
display(df.describe())

Dataset Shape (Rows, Columns): (395, 33)
--------------------------------------------------
--- FIRST 5 ROWS  ---


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


--------------------------------------------------
--- DATASET SUMMARY ---


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,16.696203,2.749367,2.521519,1.448101,2.035443,0.334177,3.944304,3.235443,3.108861,1.481013,2.291139,3.554430,5.708861,10.908861,10.713924,10.415190
std,1.276043,1.094735,1.088201,0.697505,0.839240,0.743651,0.896659,0.998862,1.113278,0.890741,1.287897,1.390303,8.003096,3.319195,3.761505,4.581443
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,3.000000,0.000000,0.000000
25%,16.000000,2.000000,2.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,8.000000,9.000000,8.000000
50%,17.000000,3.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000,11.000000,11.000000,11.000000
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,13.000000,13.000000,14.000000
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,75.000000,19.000000,19.000000,20.000000


## 🧹 Part 2: ETL Pipeline & Data Cleaning
In this section, we build an automated Extract, Transform, Load (ETL) pipeline to remove duplicate entries, handle missing data, standardize text formats, and save the polished data into the gold layer.

In [1]:
import os
import pandas as pd

print(" Starting the SmartGrad ETL Pipeline...\n")

# EXTRACT - Load the Raw Data
raw_file_path = "raw/student-mat.csv"

if not os.path.exists(raw_file_path):
    raise FileNotFoundError(
        f"Could not find '{raw_file_path}'."
    )

# Read the CSV
df = pd.read_csv(raw_file_path, sep=";")
print(f"📥 Successfully loaded raw data. Original Shape: {df.shape}")

#TRANSFORM - Clean the Data

# Remove duplicate rows
duplicate_count = df.duplicated().sum()
if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"🧹 Removed {duplicate_count} duplicate rows.")
else:
    print("✨ No duplicate rows found.")

# Handle Missing Values
null_counts = df.isnull().sum().sum()
if null_counts > 0:
    # Fill numerical columns with their average (mean)
    for col in df.select_dtypes(include=["int64", "float64"]).columns:
        df[col] = df[col].fillna(df[col].mean())

    # Fill text/categorical columns with the most common value (mode)
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].fillna(df[col].mode()[0])
    print(f"🛠️ Handled missing values across columns.")
else:
    print("✨ No missing values found in the dataset.")

# Data Type Conversions
text_columns = df.select_dtypes(include=["object"]).columns
df[text_columns] = df[text_columns].astype(str)

print("✅ Data cleaning and conversions complete.")

#LOAD - Save to Gold Folder
gold_folder = "gold"
if not os.path.exists(gold_folder):
    os.makedirs(gold_folder)
    print(f"📂 Created new folder: '{gold_folder}/'")

# Save the polished data as a standard comma-separated CSV file
clean_file_path = os.path.join(gold_folder, "clean_data.csv")
df.to_csv(clean_file_path, index=False)

print(f"\n💾 Cleaned data successfully saved to: {clean_file_path}")
print(f"🏁 Final Data Shape: {df.shape}")
print("\n🎉 ETL Pipeline executed successfully!")

 Starting the SmartGrad ETL Pipeline...

📥 Successfully loaded raw data. Original Shape: (395, 33)
✨ No duplicate rows found.
✨ No missing values found in the dataset.
✅ Data cleaning and conversions complete.
📂 Created new folder: 'gold/'

💾 Cleaned data successfully saved to: gold\clean_data.csv
🏁 Final Data Shape: (395, 33)

🎉 ETL Pipeline executed successfully!


## 🗄️ Part 3: Relational Database Schema (SQLite)
In this section, we decompose our clean flat-file into a lossless Star Schema using SQLite, enforcing strict data validation through CHECK constraints.

In [3]:
import sqlite3
import os
import pandas as pd

print("📁 Loading clean data from gold folder...")
clean_data_path = "gold/clean_data.csv"

if not os.path.exists(clean_data_path):
    raise FileNotFoundError(f"❌ Missing file: {clean_data_path}. Please run your ETL pipeline cell first!")

# Load our clean data
df = pd.read_csv(clean_data_path)

# Establish connection to SQLite database file
db_path = "student_records.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("🔨 Dropping old tables to ensure a fresh build...")
cursor.executescript("""
    DROP TABLE IF EXISTS STUDENT_PERFORMANCE;
    DROP TABLE IF EXISTS STUDENT_LIFESTYLE;
    DROP TABLE IF EXISTS STUDENT_BACKGROUND;
    DROP TABLE IF EXISTS STUDENT_CORE;
""")

# 1. CREATE TABLES
print("📐 Creating database tables with relational schemas...")
cursor.executescript("""
    CREATE TABLE STUDENT_CORE (
        student_id INTEGER PRIMARY KEY AUTOINCREMENT,
        school TEXT,
        sex TEXT,
        age INTEGER CHECK (age BETWEEN 15 AND 22),
        address TEXT,
        famsize TEXT,
        Pstatus TEXT
    );

    CREATE TABLE STUDENT_BACKGROUND (
        student_id INTEGER PRIMARY KEY,
        Medu INTEGER CHECK (Medu BETWEEN 0 AND 4),
        Fedu INTEGER CHECK (Fedu BETWEEN 0 AND 4),
        Mjob TEXT,
        Fjob TEXT,
        reason TEXT,
        guardian TEXT,
        schoolsup TEXT,
        famsup TEXT,
        FOREIGN KEY (student_id) REFERENCES STUDENT_CORE (student_id)
    );

    CREATE TABLE STUDENT_LIFESTYLE (
        student_id INTEGER PRIMARY KEY,
        traveltime INTEGER,
        studytime INTEGER CHECK (studytime BETWEEN 1 AND 4),
        activities TEXT,
        nursery TEXT,
        higher TEXT,
        internet TEXT,
        romantic TEXT,
        famrel INTEGER,
        freetime INTEGER,
        goout INTEGER,
        Dalc INTEGER,
        Walc INTEGER,
        health INTEGER,
        FOREIGN KEY (student_id) REFERENCES STUDENT_CORE (student_id)
    );

    CREATE TABLE STUDENT_PERFORMANCE (
        student_id INTEGER PRIMARY KEY,
        failures INTEGER,
        absences INTEGER,
        G1 INTEGER,
        G2 INTEGER,
        G3 INTEGER CHECK (G3 BETWEEN 0 AND 20),
        FOREIGN KEY (student_id) REFERENCES STUDENT_CORE (student_id)
    );
""")
conn.commit()

# 2. POPULATE TABLES (Fixed case-sensitivity for Pstatus)
print("📥 Migrating clean data into relational SQL tables...")

for index, row in df.iterrows():
    cursor.execute("""
        INSERT INTO STUDENT_CORE (school, sex, age, address, famsize, Pstatus)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (row['school'], row['sex'], int(row['age']), row['address'], row['famsize'], row['Pstatus']))
    
    assigned_id = cursor.lastrowid
    
    cursor.execute("""
        INSERT INTO STUDENT_BACKGROUND (student_id, Medu, Fedu, Mjob, Fjob, reason, guardian, schoolsup, famsup)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (assigned_id, int(row['Medu']), int(row['Fedu']), row['Mjob'], row['Fjob'], row['reason'], row['guardian'], row['schoolsup'], row['famsup']))
    
    cursor.execute("""
        INSERT INTO STUDENT_LIFESTYLE (student_id, traveltime, studytime, activities, nursery, higher, internet, romantic, famrel, freetime, goout, Dalc, Walc, health)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (assigned_id, int(row['traveltime']), int(row['studytime']), row['activities'], row['nursery'], row['higher'], row['internet'], row['romantic'], int(row['famrel']), int(row['freetime']), int(row['goout']), int(row['Dalc']), int(row['Walc']), int(row['health'])))
    
    cursor.execute("""
        INSERT INTO STUDENT_PERFORMANCE (student_id, failures, absences, G1, G2, G3)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (assigned_id, int(row['failures']), int(row['absences']), int(row['G1']), int(row['G2']), int(row['G3'])))

conn.commit()

# 3. VERIFICATION
print("\n🔍 Validating Data Entry Counts:")
for table in ["STUDENT_CORE", "STUDENT_BACKGROUND", "STUDENT_LIFESTYLE", "STUDENT_PERFORMANCE"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f" ✅ Table {table}: {count} records successfully migrated.")

conn.close()
print("\n🎉 Database generation complete! 'student_records.db' is ready.")

📁 Loading clean data from gold folder...
🔨 Dropping old tables to ensure a fresh build...
📐 Creating database tables with relational schemas...
📥 Migrating clean data into relational SQL tables...

🔍 Validating Data Entry Counts:
 ✅ Table STUDENT_CORE: 395 records successfully migrated.
 ✅ Table STUDENT_BACKGROUND: 395 records successfully migrated.
 ✅ Table STUDENT_LIFESTYLE: 395 records successfully migrated.
 ✅ Table STUDENT_PERFORMANCE: 395 records successfully migrated.

🎉 Database generation complete! 'student_records.db' is ready.
